In [20]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, chi2, mutual_info_classif, f_classif
from sklearn.preprocessing import MinMaxScaler

import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, pearsonr

## Sử dụng HEAT map

In [ ]:
# Giả sử bạn có một DataFrame với 78 đặc trưng
dff = pd.read_csv("data/CICDDOS2019/csv/_merged_output.csv")
df  = dff.drop(columns=["Label"])

# Tính toán ma trận tương quan
corr_matrix = df.corr()

# Vẽ heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, fmt=".2f", linewidths=0.5)
plt.title("Heatmap của Ma trận Tương quan")
plt.show()

In [16]:
# Xác định các cặp tương quan cao
threshold = 0.9
upper_triangle = np.triu(np.ones(corr_matrix.shape), k=1)
high_corr_pairs = (corr_matrix.abs() > threshold) & (upper_triangle == 1)

# Danh sách các cột cần loại bỏ
to_drop = set()
for col in high_corr_pairs.columns:
    if any(high_corr_pairs[col]):
        to_drop.add(col)

# Loại bỏ các cột tương quan cao
df_reduced = df.drop(columns=to_drop)

print(f"Số đặc trưng còn lại sau khi loại bỏ: {df_reduced.shape[1]}")
print("Các đặc trưng còn lại sau khi loại bỏ:")
print(df_reduced.columns.tolist())


Số đặc trưng còn lại sau khi loại bỏ: 49
Các đặc trưng còn lại sau khi loại bỏ:
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', '

## Chi-Square, Mutual Information, ANOVA (F-test)

In [ ]:


# 1️⃣ Đọc dữ liệu
df = pd.read_csv("data/CICDDOS2019/csv/_merged_output.csv")

# Loại bỏ các cột không chứa thông tin hữu ích
df = df.drop(columns=["Flow ID", "Source IP", "Destination IP", "Timestamp"], errors="ignore")

# Chuẩn bị dữ liệu
X = df.drop(columns=["Label"])
y = df["Label"]

# 2️⃣ Loại bỏ đặc trưng có phương sai thấp
var_thresh = VarianceThreshold(threshold=0.01)
X_var = var_thresh.fit_transform(X)
selected_var_features = X.columns[var_thresh.get_support()]

# 3️⃣ Loại bỏ đặc trưng có tương quan cao với nhau
corr_matrix = X[selected_var_features].corr(method="spearman")  # Hoặc method="pearson" khi dữ liệu tuyến tính (tắng liên tục hoặc giảm liên tục)
high_corr_features = set()
corr_threshold = 0.9  # Ngưỡng loại bỏ tương quan cao

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > corr_threshold:
            high_corr_features.add(corr_matrix.columns[i])

selected_corr_features = [f for f in selected_var_features if f not in high_corr_features]

# 4️⃣ Tính điểm Chi-Square, Mutual Information và ANOVA (F-test)
X_scaled = MinMaxScaler().fit_transform(X[selected_corr_features])  # Chuẩn hóa dữ liệu

chi_scores, _ = chi2(X_scaled, y)
mi_scores = mutual_info_classif(X_scaled, y)
anova_scores, _ = f_classif(X_scaled, y)


# sau khi có điểm Chi-Square, Mutual Information, ANOVA (F-test), chuẩn hóa, tính tổng trung bình, sắp xếp theo độ quan trọng và lấy các đặc trưng lớn hơn ngưỡng
# 5️⃣ Tạo DataFrame chứa điểm số
feature_scores = pd.DataFrame({
    "Feature": selected_corr_features,
    "Chi-Square": chi_scores,
    "Mutual Information": mi_scores,
    "ANOVA (F-test)": anova_scores
})

# 6️⃣ Chuẩn hóa điểm số để so sánh dễ dàng
for col in ["Chi-Square", "Mutual Information", "ANOVA (F-test)"]:
    feature_scores[col] = feature_scores[col] / feature_scores[col].sum()

# 7️⃣ Tính điểm tổng hợp bằng trung bình cộng
feature_scores["Total Score"] = feature_scores[["Chi-Square", "Mutual Information", "ANOVA (F-test)"]].mean(axis=1)

# Sắp xếp theo độ quan trọng
feature_scores = feature_scores.sort_values(by="Total Score", ascending=False)

# 8️⃣ Lọc đặc trưng tốt nhất: chỉ giữ lại những đặc trưng có điểm > 1% tổng trọng số
threshold = 0.01
selected_features = feature_scores[feature_scores["Total Score"] > threshold]["Feature"].tolist()

# In ra danh sách đặc trưng còn lại
print(f"Số đặc trưng tốt nhất sau khi áp dụng tất cả phương pháp: {len(selected_features)}")
print("Danh sách đặc trưng được chọn:", selected_features)


Số đặc trưng tốt nhất sau khi áp dụng tất cả phương pháp: 24
Danh sách đặc trưng được chọn: ['Fwd Packet Length Min', 'ACK Flag Count', 'Flow Bytes/s', 'URG Flag Count', 'Protocol', 'Fwd Packet Length Max', 'Flow Duration', 'Fwd Packets Length Total', 'CWE Flag Count', 'Init Fwd Win Bytes', 'Packet Length Std', 'Idle Std', 'Init Bwd Win Bytes', 'Fwd PSH Flags', 'Fwd Header Length', 'Down/Up Ratio', 'Fwd Packet Length Std', 'Bwd Packets/s', 'Bwd Packet Length Min', 'Total Fwd Packets', 'Bwd Packets Length Total', 'Bwd IAT Std', 'Fwd Seg Size Min', 'Bwd Packet Length Std']
